In [163]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [164]:
import json
from pathlib import Path

import ollama

from src.data_loader import DataLoader
from src.models import Question, Answer, GradingResult, KeyConcepts

In [165]:
avail_data = [DataLoader(question_dir) for question_dir in Path("../data").iterdir() if question_dir.is_dir() ]

In [ ]:
def get_system_prompt(question: Question) -> str:
    return f"""
Ты — эксперт по выделению проверяемых смысловых блоков из эталонных ответов.

Задача:
На основе эталонного ответа преподавателя сформулируй список ключевых утверждений (смысловых блоков), которые должен раскрыть студент, чтобы ответ считался полным.

Правила:
- Каждый блок должен быть конкретным фактом, а не обобщённой категорией.
- Формулируй блок как законченное утверждение, которое можно однозначно проверить в ответе студента.
- Не копируй фразы дословно, но сохраняй все ключевые детали. Допускай синонимы и перефразировки.
- Если несколько связанных фактов относятся к одной идее — объедини их в один блок.
- Не привязывайся к точным числам, если они не являются сутью. Допускай округление или обобщение временных/количественных значений.
- Количество блоков определи самостоятельно, исходя из содержания эталона. Не задавай фиксированное число.
- Для каждого блока определи ОТНОСИТЕЛЬНЫЙ вес, его важность в контексте этого вопроса и эталонного ответа.

Формат вывода: JSON-список строк.
"""

In [167]:
def get_prompt(question: Question):
    return f"""
Вопрос:
{question.text}

Эталонный ответ:
{question.reference_answer}
"""

In [168]:
models = {
    "qwen": "qwen2.5:7b",
    "saiga": "bambucha/saiga-llama3:8b",
    "vikhr": "rscr/vikhr_llama3.1_8b:Q4_K_M",
    "llama": "llama3:8b-instruct-q4_K_M",
    "mistral": "mistral:7b"
}

In [169]:
def generate_key_concepts(question: Question, model: str, temperature: float):
    response = ollama.generate(
        model=model,
        system=get_system_prompt(question),
        prompt=get_prompt(question),
        format=KeyConcepts.model_json_schema(),
        options={"temperature": temperature},
    )

    if not response.response:
        raise ollama.ResponseError("Response is None")
    
    if response.eval_duration:
        print(f"DONE in {(response.eval_duration / 1_000_000_000):.2f} s")
    
    return KeyConcepts.model_validate(json.loads(response.response))

In [170]:
question = avail_data[2].question

In [171]:
result = generate_key_concepts(question, models["qwen"], 0.8)

DONE in 8.24 s


In [172]:
print(f"Вопрос: {question.text}")
print(f"Эталонный ответ: {question.reference_answer}")
print(f"-------------------")
print(f"Выделенные понятия:")
for concept in result.concepts:
    print(concept)

Вопрос: Объясните механизм внимания (Attention) в архитектуре трансформеров. Как работает self-attention и почему он важен?
Эталонный ответ: Механизм внимания (Attention) в трансформерах позволяет модели динамически фокусироваться на различных частях входной последовательности при обработке каждого элемента. Self-attention (или intra-attention) вычисляет взаимосвязи между всеми позициями в последовательности одновременно.

Механизм работает следующим образом: для каждого элемента входной последовательности создаются три вектора — Query (запрос), Key (ключ) и Value (значение). Эти векторы получаются путем линейного преобразования входного эмбеддинга через обучаемые весовые матрицы W_Q, W_K, W_V.

Формула scaled dot-product attention: Attention(Q, K, V) = softmax(Q·K^T / √d_k) · V, где d_k — размерность ключей. Масштабирование на √d_k предотвращает проблему затухающих градиентов при больших значениях d_k.

Multi-head attention расширяет этот механизм, позволяя модели одновременно attend 